<style>
.reveal { font-family: 'Segoe UI', system-ui, sans-serif; }
.reveal h2 { color: #0D2240; border-bottom: 2.5px solid #1A7A9A;
             padding-bottom:.15em; margin-bottom:.5em; }
.reveal .slides section { text-align: left; }
.reveal pre { font-size:.62em; border-radius:6px; border:1px solid #E2E6EE; box-shadow:none; }
.reveal pre code { max-height:420px; }
.reveal ul li, .reveal ol li { margin-bottom:.3em; }
.defn { background:#EAF6FA; border-left:4px solid #1A7A9A;
        padding:.6em 1em; border-radius:0 6px 6px 0; margin:.5em 0; }
.nota { background:#FFF8E1; border-left:4px solid #C8961E;
        padding:.6em 1em; border-radius:0 6px 6px 0; margin:.5em 0; }
</style>

## Pensamiento Sistémico e Introducción a la Simulación
### Modelado de Sistemas bajo Incertidumbre · Semana 1
---
Departamento de Ingeniería Industrial · Universidad de los Andes

## Agenda
1. ¿Qué es un sistema?
2. Sistemas determinísticos vs. estocásticos
3. La falacia del promedio
4. ¿Qué es simular? ¿Cuándo es útil?
5. Tipos de simulación
6. Ejemplo motivador: el vendedor de periódicos

## ¿Qué es un sistema?
<div class="defn">

Un **sistema** es un conjunto de componentes que interactúan entre sí para lograr un objetivo común.

</div>

**Ejemplos en ingeniería industrial:**
- Línea de manufactura: máquinas, operarios, inventario
- Centro de distribución: camiones, bodegas, pedidos
- Hospital: médicos, pacientes, camas, equipos
- Red de transporte: vehículos, rutas, demanda

## Componentes de un sistema
| Elemento | Descripción | Ejemplo (banco) |
|---|---|---|
| **Entidades** | Objetos de interés | Clientes |
| **Atributos** | Características de entidades | Tipo de transacción |
| **Estado** | Variables que describen el sistema | # clientes en cola |
| **Eventos** | Cambios instantáneos de estado | Llegada, inicio servicio |
| **Actividades** | Procesos con duración | Servicio en caja |

## Sistemas Determinísticos vs. Estocásticos
| | **Determinístico** | **Estocástico** |
|---|---|---|
| Entradas | Fijas y conocidas | Aleatorias |
| Salidas | Únicas y predecibles | Distribuciones de resultados |
| Herramienta | Optimización, álgebra | Simulación, probabilidad |
| Ejemplo | Ruta óptima (distancias fijas) | Tiempos de llegada de camiones |

<div class="nota">
La mayoría de los sistemas reales son <strong>estocásticos</strong>: los tiempos de llegada, las fallas, la demanda y los tiempos de servicio varían aleatoriamente.
</div>

## La Falacia del Promedio
**Pregunta:** Un puente soporta 10 toneladas. El camión promedio pesa 8 t. ¿Es seguro?

*Intuitivamente: sí. En la realidad: depende.*

In [ ]:
import numpy as np, matplotlib.pyplot as plt
np.random.seed(42)
pesos = np.random.normal(loc=8, scale=1.5, size=10000)
fig, ax = plt.subplots(figsize=(8,3))
ax.hist(pesos, bins=50, color='#1A7A9A', edgecolor='white', alpha=0.85)
ax.axvline(10, color='#C8961E', lw=2, ls='--', label='Límite (10 t)')
ax.axvline(8,  color='#0D2240', lw=2, ls='-',  label='Promedio (8 t)')
ax.set_xlabel('Peso del camión (t)'); ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de pesos de camiones')
ax.legend(); plt.tight_layout(); plt.show()
prob = np.mean(pesos > 10)
print(f"Probabilidad de exceder la capacidad: {prob:.1%}")

## ¿Qué es la Simulación?
<div class="defn">

**Simular** es imitar el comportamiento de un sistema real a lo largo del tiempo, usando un modelo computacional que captura la aleatoriedad del sistema.

</div>

**La simulación permite:**
- Experimentar sin interrumpir el sistema real
- Evaluar escenarios hipotéticos ("¿qué pasa si…?")
- Estimar el desempeño bajo incertidumbre
- Comparar alternativas de diseño

## ¿Cuándo simular?
**Simular es apropiado cuando:**
- El sistema tiene **aleatoriedad** significativa
- El sistema es **demasiado complejo** para modelos analíticos exactos
- Se requiere **experimentar** sin costos ni riesgos reales
- Se necesita **estudiar el comportamiento dinámico**

**Simular NO es apropiado cuando:**
- Existe una solución analítica exacta y accesible
- El problema puede resolverse con optimización directa
- Los datos de entrada son insuficientes para calibrar el modelo

## Tipos de Simulación
| Tipo | Descripción | Ejemplo |
|---|---|---|
| **Monte Carlo** | Muestreo aleatorio estático | Estimación de π, análisis de riesgo |
| **Eventos Discretos (SED)** | Sistema evoluciona por eventos | Colas, manufactura, logística |
| **Dinámica de Sistemas** | Flujos y stocks continuos | Cadenas de suministro, epidemias |
| **Basada en Agentes** | Entidades autónomas que interactúan | Tráfico, mercados |

*En este curso: **Monte Carlo** y **Simulación de Eventos Discretos***.  

## Ejemplo: El Vendedor de Periódicos
**Problema:** ¿Cuántos periódicos comprar cada mañana?
- Costo de compra: \$1.000 por unidad
- Precio de venta: \$1.500 por unidad
- Demanda diaria: desconocida (aleatoria)
- Sobrantes: sin valor

**Modelo determinístico:** comprar el promedio → subóptimo  
**Modelo estocástico:** simular múltiples escenarios → decisión robusta

In [ ]:
import numpy as np
np.random.seed(0)
demanda = np.random.randint(40, 81, size=10000)  # demanda entre 40 y 80

def ganancia(q, d, c=1000, p=1500):
    ventas = min(q, d)
    return ventas * p - q * c

cantidades = range(40, 81)
ganancias_esperadas = [np.mean([ganancia(q, d) for d in demanda]) for q in cantidades]

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,3))
ax.plot(list(cantidades), ganancias_esperadas, color='#1A7A9A', lw=2)
ax.axvline(cantidades[np.argmax(ganancias_esperadas)],
           color='#C8961E', ls='--', lw=2,
           label=f"Óptimo: {cantidades[np.argmax(ganancias_esperadas)]} unidades")
ax.set_xlabel('Unidades compradas'); ax.set_ylabel('Ganancia esperada ($)')
ax.set_title('Ganancia esperada vs. cantidad comprada')
ax.legend(); plt.tight_layout(); plt.show()

## Conclusiones
- Los sistemas reales son **estocásticos**: la aleatoriedad importa
- La **falacia del promedio** puede llevar a decisiones subóptimas o peligrosas
- La simulación es una herramienta poderosa para **analizar sistemas complejos bajo incertidumbre**
- En este curso: Monte Carlo + Eventos Discretos + Cadenas de Markov

<div class="nota">
<strong>Prerequisito:</strong> Las lecturas de cada semana deben revisarse <em>antes</em> de la clase.  La clase es para discusión y aplicación, no para presentar los conceptos desde cero.
</div>